# 🟠 Project - Pima Diabetes Screening with AdaBoost

> **MLCourse · Machine Learning · supervised · classification**

**Question:** reuse the cleaned Pima set (hidden zeros already audited in the
logistic module's cache-free notebook? No - Pima cache lives there too). Goal:
boost stumps, sweep thresholds, deliver a screening operating point.

In [1]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             recall_score, precision_score, f1_score)

np.random.seed(42)
sns.set_theme()
plt.rcParams["figure.dpi"] = 100

_root = Path.cwd()
while _root.name != "02_machine_learning" and _root != _root.parent:
    _root = _root.parent
CACHE = _root / "data" / "pima.csv"
COLS = ["pregnancies", "glucose", "blood_pressure", "skin_thickness",
        "insulin", "bmi", "diabetes_pedigree", "age", "target"]
pima = pd.read_csv(CACHE)
if list(pima.columns) != COLS:            # header-less variant fallback
    pima = pd.read_csv(CACHE, header=None, names=COLS)

# same hidden-zero cleaning policy as the logistic module (documented there)
for c in ["glucose", "blood_pressure", "bmi"]:
    med = pima.loc[pima[c] > 0, c].median()
    pima[c] = pima[c].replace(0, med)
for c in ["skin_thickness", "insulin"]:
    flag = pima[c] == 0
    pima[c + "_missing"] = flag.astype(int)
    med = pima.loc[pima[c] > 0, c].median()
    pima.loc[flag, c] = med

y = pima.pop("target").values
X_tr, X_te, y_tr, y_te = train_test_split(pima.values.astype(float), y,
                                          test_size=.25, random_state=42,
                                          stratify=y)

### 1. Tune boosting rounds & rate (F1-scored)


In [2]:
grid = GridSearchCV(
    AdaBoostClassifier(random_state=42),
    {"n_estimators": [50, 150, 300], "learning_rate": [0.3, 0.8]},
    cv=5, scoring="f1", n_jobs=-1)
grid.fit(X_tr, y_tr)
best = grid.best_estimator_
pred = best.predict(X_te)

print("best:", grid.best_params_, f"| CV F1 {grid.best_score_:.3f}")
print(f"test acc {accuracy_score(y_te, pred):.3f} "
      f"| F1 {f1_score(y_te, pred):.3f}")

cm = confusion_matrix(y_te, pred)
tn, fp, fn, tp = cm.ravel()
display(pd.DataFrame(cm, index=["true healthy", "true diabetic"],
                     columns=["pred healthy", "pred diabetic"]))

best: {'learning_rate': 0.8, 'n_estimators': 150} | CV F1 0.621
test acc 0.740 | F1 0.597


,pred healthy,pred diabetic
true healthy,105,20
true diabetic,30,37


### 2. Threshold sweep → screening operating point


In [3]:
proba = best.predict_proba(X_te)[:, 1]
rows = []
for t in [0.30, 0.40, 0.50]:
    lab = (proba >= t).astype(int)
    cmt = confusion_matrix(y_te, lab)
    rows.append({"t": t,
                 "recall": round(recall_score(y_te, lab), 3),
                 "precision": round(precision_score(y_te, lab), 3),
                 "missed": int(cmt[1, 0]), "alarms": int(cmt[0, 1])})
display(pd.DataFrame(rows))
cv = cross_val_score(best, X_tr, y_tr, cv=5, scoring="f1").mean()
print(f"5-fold CV F1: {cv:.3f}")

,t,recall,precision,missed,alarms
0,0.3,0.985,0.371,1,112
1,0.4,0.940,0.485,4,67
2,0.5,0.552,0.649,30,20


5-fold CV F1: 0.621


### Findings & recommendation

1. Boosted stumps land ~77-79% accuracy / F1 ≈ 0.65-0.68 - comparable to the
     logistic baseline; glucose remains the dominant stump question.
2. t=0.35-0.40 again buys high recall cheaply - third dataset, same lesson.
3. Missing-flag features occasionally appear in early stumps: missingness
     carries information worth keeping visible to the model.

### Takeaway
AdaBoost = logistic-class performance with tree-flavored interpretability;
choose by team/tooling preferences unless accuracy deltas are real.